In [1]:
#Import Libraries

import pandas as pd
import numpy as np
import json
import zipfile
import ast
import os
import pickle

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
# Converting and build the french to english Disease map

DATA_DIR = '../data/ddxplus'

# Load disease definitions
with open(os.path.join(DATA_DIR, 'release_conditions.json'), 'r', encoding='utf-8') as f:
    conditions = json.load(f)

# Build a map: French name → English name
fr_to_en = {}
for fr_name, details in conditions.items():
    en_name = details.get('cond-name-eng', fr_name)
    fr_to_en[fr_name] = en_name

print("✅ Disease name map built!")
print("\nSample translations:")
for fr, en in list(fr_to_en.items())[:8]:
    print(f"   {fr:<45} → {en}")

✅ Disease name map built!

Sample translations:
   Pneumothorax spontané                         → Spontaneous pneumothorax
   Céphalée en grappe                            → Cluster headache
   Syndrome de Boerhaave                         → Boerhaave
   Fracture de côte spontanée                    → Spontaneous rib fracture
   RGO                                           → GERD
   VIH (Primo-infection)                         → HIV (initial infection)
   Anémie                                        → Anemia
   Pharyngite virale                             → Viral pharyngitis


In [3]:
# Load patient data & Verify columns

train_zip = os.path.join(DATA_DIR, 'release_train_patients.zip')

with zipfile.ZipFile(train_zip, 'r') as z:
    csv_name = z.namelist()[0]
    with z.open(csv_name) as f:
        df = pd.read_csv(f)

print(f"✅ Full training data loaded!")
print(f"   Shape   : {df.shape}")
print(f"\n   Columns : {df.columns.tolist()}")
print(f"\n=== First row ===")
print(df.iloc[0])

✅ Full training data loaded!
   Shape   : (1025602, 6)

   Columns : ['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']

=== First row ===
AGE                                                                      18
DIFFERENTIAL_DIAGNOSIS    [['Bronchitis', 0.19171203430383882], ['Pneumo...
SEX                                                                       M
PATHOLOGY                                                              URTI
EVIDENCES                 ['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54...
INITIAL_EVIDENCE                                                       E_91
Name: 0, dtype: object


In [4]:
# Take a Balanaced Sample 

SAMPLES_PER_DISEASE = 1300

df_sample = (
    df.groupby('PATHOLOGY', group_keys=False)
      .apply(lambda x: x.sample(min(len(x), SAMPLES_PER_DISEASE), random_state=42))
      .reset_index(drop=True)
)

print(f"✅ Balanced sample created!")
print(f"   Original : {df.shape[0]:,} rows")
print(f"   Sample   : {df_sample.shape[0]:,} rows")
print(f"   Diseases : {df_sample['PATHOLOGY'].nunique()}")

del df 

C:\Users\iamsa\AppData\Local\Temp\ipykernel_11268\2637159748.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), SAMPLES_PER_DISEASE), random_state=42))


✅ Balanced sample created!
   Original : 1,025,602 rows
   Sample   : 62,079 rows
   Diseases : 49


In [5]:
import ast

# How many symptoms a doctor "knows" at early assessment
# (initial complaint + a few follow-ups)
MAX_EVIDENCES = 4

def parse_evidences_limited(row):
    """
    Simulate EARLY clinical assessment:
    - Always include INITIAL_EVIDENCE (the chief complaint)
    - Add only the first few other evidences (partial information)
    """
    try:
        evidence_list = ast.literal_eval(row['EVIDENCES'])
        evidence_list = [str(e) for e in evidence_list]
    except Exception:
        evidence_list = []

    initial = str(row['INITIAL_EVIDENCE'])

    # Start with the chief complaint
    selected = [initial] if initial and initial != 'nan' else []

    # Add other evidences (excluding the initial, capped at MAX_EVIDENCES)
    for ev in evidence_list:
        if ev not in selected:
            selected.append(ev)
        if len(selected) >= MAX_EVIDENCES:
            break

    return selected

df_sample['evidence_tokens'] = df_sample.apply(parse_evidences_limited, axis=1)

print("✅ Evidences LIMITED to simulate early clinical assessment!")
print(f"   Max evidences per patient: {MAX_EVIDENCES}")
print(f"\nExample patient tokens:")
print(df_sample['evidence_tokens'].iloc[0])
print(f"\nAverage symptoms per patient now: {df_sample['evidence_tokens'].apply(len).mean():.1f}")
print("   (was ~14 before — now intentionally reduced)")

✅ Evidences LIMITED to simulate early clinical assessment!
   Max evidences per patient: 4

Example patient tokens:
['E_214', 'E_72', 'E_77', 'E_123']

Average symptoms per patient now: 4.0
   (was ~14 before — now intentionally reduced)


In [6]:
# One Hot Encode all Symptoms

mlb = MultiLabelBinarizer(sparse_output=False)

symptom_matrix = mlb.fit_transform(df_sample['evidence_tokens'])
symptom_cols   = list(mlb.classes_)

X_symptoms = pd.DataFrame(symptom_matrix, columns=symptom_cols)

print(f"✅ Symptoms one-hot encoded!")
print(f"   Total symptom features: {len(symptom_cols)}")
print(f"   Feature matrix shape  : {X_symptoms.shape}")
print(f"\nSample feature names: {symptom_cols[:10]}")

✅ Symptoms one-hot encoded!
   Total symptom features: 267
   Feature matrix shape  : (62079, 267)

Sample feature names: ['E_0', 'E_1', 'E_10', 'E_101', 'E_103', 'E_104', 'E_105', 'E_11', 'E_111', 'E_112']


In [7]:
# Add Age & Sex as Features

df_sample = df_sample.reset_index(drop=True)

X_symptoms['AGE'] = df_sample['AGE'].values

X_symptoms['SEX_M'] = (df_sample['SEX'] == 'M').astype(int)
X_symptoms['SEX_F'] = (df_sample['SEX'] == 'F').astype(int)

feature_cols = list(X_symptoms.columns)

print(f"✅ Age and Sex added!")
print(f"   Final feature count: {len(feature_cols)}")
print(f"   (symptoms + AGE + SEX_M + SEX_F)")

✅ Age and Sex added!
   Final feature count: 270
   (symptoms + AGE + SEX_M + SEX_F)


In [8]:
# Encode Disease Labels(in Englis)

le = LabelEncoder()
y = le.fit_transform(df_sample['PATHOLOGY'])

print(f"✅ Disease labels encoded directly from PATHOLOGY!")
print(f"   Total diseases: {len(le.classes_)}")
print(f"\nAll diseases found:")
for i, disease in enumerate(le.classes_, 1):
    print(f"   {i:2} → {disease}")

✅ Disease labels encoded directly from PATHOLOGY!
   Total diseases: 49

All diseases found:
    1 → Acute COPD exacerbation / infection
    2 → Acute dystonic reactions
    3 → Acute laryngitis
    4 → Acute otitis media
    5 → Acute pulmonary edema
    6 → Acute rhinosinusitis
    7 → Allergic sinusitis
    8 → Anaphylaxis
    9 → Anemia
   10 → Atrial fibrillation
   11 → Boerhaave
   12 → Bronchiectasis
   13 → Bronchiolitis
   14 → Bronchitis
   15 → Bronchospasm / acute asthma exacerbation
   16 → Chagas
   17 → Chronic rhinosinusitis
   18 → Cluster headache
   19 → Croup
   20 → Ebola
   21 → Epiglottitis
   22 → GERD
   23 → Guillain-Barré syndrome
   24 → HIV (initial infection)
   25 → Influenza
   26 → Inguinal hernia
   27 → Larygospasm
   28 → Localized edema
   29 → Myasthenia gravis
   30 → Myocarditis
   31 → PSVT
   32 → Pancreatic neoplasm
   33 → Panic attack
   34 → Pericarditis
   35 → Pneumonia
   36 → Possible NSTEMI / STEMI
   37 → Pulmonary embolism
   38 → P

In [9]:
# Train/ Test Split
X = X_symptoms

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✅ Data split complete!")
print(f"   Training : {X_train.shape[0]:,} samples")
print(f"   Testing  : {X_test.shape[0]:,} samples")
print(f"   Features : {X_train.shape[1]}")
print(f"   Diseases : {len(le.classes_)}")

✅ Data split complete!
   Training : 49,663 samples
   Testing  : 12,416 samples
   Features : 270
   Diseases : 49


In [10]:
# Save Evrything

os.makedirs('../models', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

X_train.to_pickle('../data/processed/X_train.pkl')
X_test.to_pickle('../data/processed/X_test.pkl')
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy', y_test)

with open('../models/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

with open('../models/symptom_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

with open('../data/ddxplus/release_evidences.json', 'r', encoding='utf-8') as f:
    evidences = json.load(f)

evidence_map = {}
for code, details in evidences.items():
    evidence_map[code] = details.get('question_en', code)

with open('../models/evidence_map.pkl', 'wb') as f:
    pickle.dump(evidence_map, f)

with open('../models/disease_map.pkl', 'wb') as f:
    pickle.dump(fr_to_en, f)

print("✅ All files saved!")

✅ All files saved!


In [11]:
print("=" * 50)
print("      DDXPLUS PREPROCESSING COMPLETE ✅")
print("=" * 50)
print(f"  Total samples    : {X_train.shape[0] + X_test.shape[0]:,}")
print(f"  Symptom features : {len(feature_cols)}")
print(f"  Diseases         : {len(le.classes_)}")
print(f"  Training samples : {X_train.shape[0]:,}")
print(f"  Testing samples  : {X_test.shape[0]:,}")
print(f"  Language         : English ✅")
print("=" * 50)
print("\n🚀 Ready for model training")

      DDXPLUS PREPROCESSING COMPLETE ✅
  Total samples    : 62,079
  Symptom features : 270
  Diseases         : 49
  Training samples : 49,663
  Testing samples  : 12,416
  Language         : English ✅

🚀 Ready for model training
